# GAT-Based Cell Clustering for scRNA-Seq Data

Comparing Graph Attention Network (GAT) embedding refinement against KMeans and SC3 consensus clustering on two single-cell RNA-seq datasets, under clean and dropout-corrupted conditions.

| Accession | Biological Source | Cells | Cell Types | 
| :--- | :--- | :--- | :--- |
| **GSE65525** (Klein) | Mouse Embryonic Stem Cells | 2,717 | 4 stages (d0, d2, d4, d7) |
| **GSE60361** (Zeisel) | Mouse Cerebral Cortex | 3,005 | 7 cell types |


In [ ]:
# ============================================================
# Section 1: Setup
# ============================================================

import copy
import gzip
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sc3s
import scanpy as sc
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
from torch_geometric.nn import GATConv

INPUT_DIR = Path("data/input")
OUTPUT_DIR = Path("data/output")


## 2. Data Loading


In [ ]:
# ============================================================
# Load Klein (GSE65525) - Embryonic Stem Cells
# Downloads from GEO if not already present
# ============================================================

def fetch_file(url: str, dest: Path):
    if not dest.exists() or dest.stat().st_size == 0:
        print(f"Downloading {dest.name}...")
        try:
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            if dest.exists(): dest.unlink()
            raise e

def get_GSE65525():
    tar_file = Path("GSE65525_RAW.tar")
    extract_dir = Path("./GSE65525_extracted")

    if not tar_file.exists() or tar_file.stat().st_size == 0:
        print(f"Downloading {tar_file}...")
        urllib.request.urlretrieve(
            "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE65525&format=file", tar_file
        )

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tar_file, "r") as tar:
            tar.extractall(path=extract_dir)

    print("Parsing dataset files...")
    dfs, obs_data = [], {}

    mouse_files = sorted([
        f for f in extract_dir.glob("*GSM159949*")
        if f.name.endswith(('.csv', '.csv.gz', '.csv.bz2'))
    ])

    for i, filepath in enumerate(mouse_files):
        df_t = pd.read_csv(filepath, index_col=0).T
        df_t.index = [f"b{i}_{cid}" for cid in df_t.index]
        stage = filepath.name.split("_")[2] if len(filepath.name.split("_")) > 2 else "unknown"
        for cid in df_t.index:
            obs_data[cid] = stage
        dfs.append(df_t)

    expr_df = pd.concat(dfs)
    expr_df.columns.name, expr_df.index.name = None, None

    adata = sc.AnnData(
        X=expr_df.values.astype(float),
        obs=pd.DataFrame({"cell_stage": obs_data}),
        var=pd.DataFrame(index=expr_df.columns)
    )
    adata.var_names_make_unique()
    return adata


klein_adata = get_GSE65525()
print(f"Klein: {klein_adata.n_obs} cells x {klein_adata.n_vars} genes")
print(f"Stages: {klein_adata.obs['cell_stage'].value_counts().to_dict()}")


In [ ]:
# ============================================================
# Load Zeisel (GSE60361) - Mouse Cerebral Cortex
# Expression file + labels from scGNN benchmark data
# ============================================================

expr_df = pd.read_csv(INPUT_DIR / "GSE60361_C1-3005-Expression.txt", sep="\t", index_col=0)
print(f"Raw expression shape: {expr_df.shape}")  # genes x cells

# Transpose so cells are obs rows
zeisel_adata = sc.AnnData(
    X=expr_df.T.values.astype(np.float32),
    obs=pd.DataFrame(index=expr_df.columns),
    var=pd.DataFrame(index=expr_df.index)
)
zeisel_adata.obs_names_make_unique()
zeisel_adata.var_names_make_unique()

# Attach cell-type labels
labels_df = pd.read_csv(INPUT_DIR / "Zeisel_cell_label.csv", index_col=0)
zeisel_adata.obs["cell_type"] = labels_df.reindex(zeisel_adata.obs_names)["Label"].values.astype(str)

print(f"Zeisel: {zeisel_adata.n_obs} cells x {zeisel_adata.n_vars} genes")
print(f"Labels: {zeisel_adata.obs['cell_type'].value_counts().to_dict()}")


## 3. Preprocessing + PCA

QC filtering, normalization, and PCA dimensionality reduction for both datasets.


In [ ]:
# ============================================================
# Preprocess Klein
# ============================================================

klein_adata.var["mt"] = klein_adata.var_names.str.lower().str.startswith("mt-")
klein_adata.var["ribo"] = klein_adata.var_names.str.startswith(("RPS", "RPL"))
sc.pp.calculate_qc_metrics(klein_adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True)

sc.pp.filter_cells(klein_adata, min_genes=100)
sc.pp.filter_genes(klein_adata, min_cells=3)
sc.pp.scrublet(klein_adata, batch_key="cell_stage")
print(f"Klein after QC: {klein_adata.n_obs} cells x {klein_adata.n_vars} genes")

klein_adata.layers["counts"] = klein_adata.X.copy()
sc.pp.normalize_total(klein_adata)
sc.pp.log1p(klein_adata)

# PCA + t-SNE
sc.tl.pca(klein_adata, n_comps=50)
sc.tl.tsne(klein_adata, n_pcs=50)
print(f"Klein PCA: {klein_adata.obsm['X_pca'].shape}")


In [ ]:
# ============================================================
# Preprocess Zeisel
# ============================================================

zeisel_adata.var["mt"] = zeisel_adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(zeisel_adata, qc_vars=["mt"], inplace=True, log1p=True)

sc.pp.filter_cells(zeisel_adata, min_genes=100)
sc.pp.filter_genes(zeisel_adata, min_cells=3)
print(f"Zeisel after QC: {zeisel_adata.n_obs} cells x {zeisel_adata.n_vars} genes")

zeisel_adata.layers["counts"] = zeisel_adata.X.copy()
sc.pp.normalize_total(zeisel_adata)
sc.pp.log1p(zeisel_adata)

# PCA + t-SNE
sc.tl.pca(zeisel_adata, n_comps=50)
sc.tl.tsne(zeisel_adata, n_pcs=50)
print(f"Zeisel PCA: {zeisel_adata.obsm['X_pca'].shape}")


## 4. Baselines — KMeans + SC3 (Clean Data)


In [ ]:
# ============================================================
# Klein baselines
# ============================================================

klein_pca = klein_adata.obsm["X_pca"][:, :50]
klein_k_values = [4, 8, 10]

# KMeans
for k in klein_k_values:
    km = KMeans(n_clusters=k, random_state=5, n_init=10)
    klein_adata.obs[f"kmeans_k{k}"] = km.fit_predict(klein_pca).astype(str)

# SC3
print("Running SC3 on Klein...")
sc3s.tl.consensus(klein_adata, n_clusters=klein_k_values)

# Visualize
sc.pl.tsne(
    klein_adata,
    color=["cell_stage"] + [f"kmeans_k{k}" for k in klein_k_values],
    ncols=4, size=2, legend_loc="on data",
    title=["Ground Truth"] + [f"KMeans (k={k})" for k in klein_k_values]
)
sc.pl.tsne(
    klein_adata,
    color=[f"sc3s_{k}" for k in klein_k_values],
    ncols=3, size=2, legend_loc="on data",
    title=[f"SC3 (k={k})" for k in klein_k_values]
)


In [ ]:
# ============================================================
# Zeisel baselines
# ============================================================

zeisel_pca = zeisel_adata.obsm["X_pca"][:, :50]
zeisel_k_values = [5, 7, 9]

# KMeans
for k in zeisel_k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    zeisel_adata.obs[f"kmeans_k{k}"] = km.fit_predict(zeisel_pca).astype(str)

# SC3
print("Running SC3 on Zeisel...")
sc3s.tl.consensus(zeisel_adata, n_clusters=zeisel_k_values)

# Visualize k=7 (matches ground truth)
sc.pl.tsne(
    zeisel_adata,
    color=["cell_type", "kmeans_k7", "sc3s_7"],
    ncols=3, size=2, legend_loc="on data",
    title=["Ground Truth (7 types)", "KMeans (k=7)", "SC3 (k=7)"]
)


## 5. GAT — Graph Attention Network

Unsupervised graph autoencoder: GAT encoder compresses node features, linear decoder reconstructs them via MSE loss. No labels used during training. Refined embeddings are then clustered with KMeans.


In [ ]:
# ============================================================
# GAT Graph Autoencoder definition
# ============================================================

class GATAutoencoder(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, embed_dim, heads=4):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=0.3)
        self.gat2 = GATConv(hidden_dim * heads, embed_dim, heads=1, dropout=0.3)
        self.decoder = torch.nn.Linear(embed_dim, in_dim)

    def encode(self, x, edge_index):
        h = F.elu(self.gat1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.gat2(h, edge_index)
        return h

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        x_recon = self.decoder(z)
        return x_recon, z


def build_knn_graph(embeddings, k=15):
    knn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    knn.fit(embeddings)
    _, indices = knn.kneighbors(embeddings)

    src, dst = [], []
    for i in range(len(embeddings)):
        for j in indices[i]:
            if i != j:
                src.append(i)
                dst.append(j)
    return torch.tensor([src, dst], dtype=torch.long)


def train_gat(pca_embeddings, k=15, hidden_dim=64, embed_dim=32, epochs=200, lr=0.005):
    """Train unsupervised GAT on PCA embeddings, return refined embeddings."""
    edge_index = build_knn_graph(pca_embeddings, k=k)
    x = torch.tensor(pca_embeddings, dtype=torch.float32)
    print(f"  Graph: {x.shape[0]} nodes, {edge_index.shape[1]} edges")

    model = GATAutoencoder(in_dim=x.shape[1], hidden_dim=hidden_dim, embed_dim=embed_dim, heads=4)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        x_recon, z = model(x, edge_index)
        loss = F.mse_loss(x_recon, x)
        loss.backward()
        optimizer.step()

        if epoch % 50 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        _, refined = model(x, edge_index)
    return refined.numpy()


print("GAT functions defined.")


In [ ]:
# ============================================================
# Run GAT on both datasets (clean)
# ============================================================

# Klein
print("Training GAT on Klein (clean)...")
klein_gat_embeddings = train_gat(klein_pca, k=15, hidden_dim=64, embed_dim=32, epochs=200)
for k in klein_k_values:
    km = KMeans(n_clusters=k, random_state=5, n_init=10)
    klein_adata.obs[f"gat_k{k}"] = km.fit_predict(klein_gat_embeddings).astype(str)

print(f"\nKlein GAT embeddings: {klein_gat_embeddings.shape}")

# Zeisel
print("\nTraining GAT on Zeisel (clean)...")
zeisel_gat_embeddings = train_gat(zeisel_pca, k=15, hidden_dim=64, embed_dim=32, epochs=200)
for k in zeisel_k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    zeisel_adata.obs[f"gat_k{k}"] = km.fit_predict(zeisel_gat_embeddings).astype(str)

print(f"Zeisel GAT embeddings: {zeisel_gat_embeddings.shape}")

# Visualize GAT results
sc.pl.tsne(
    klein_adata,
    color=["cell_stage"] + [f"gat_k{k}" for k in klein_k_values],
    ncols=4, size=2, legend_loc="on data",
    title=["Ground Truth"] + [f"GAT (k={k})" for k in klein_k_values]
)
sc.pl.tsne(
    zeisel_adata,
    color=["cell_type", "gat_k7"],
    ncols=2, size=2, legend_loc="on data",
    title=["Ground Truth", "GAT (k=7)"]
)


## 6. Dropout Experiment

Inject 15% artificial dropouts into both datasets, re-run preprocessing + all three methods. Measures robustness to noise.


In [ ]:
# ============================================================
# Inject dropouts into both datasets
# ============================================================

dropout_rate = 0.15

def inject_dropouts(adata, rate, seed):
    """Create a noisy copy with artificial dropouts on the raw count layer."""
    noisy = copy.deepcopy(adata)
    np.random.seed(seed)
    # Operate on raw counts, not log-normalized .X
    matrix = noisy.layers["counts"].copy()
    non_zero_mask = matrix > 0
    keep_mask = np.random.binomial(1, 1 - rate, size=matrix.shape).astype(bool)
    matrix[non_zero_mask & ~keep_mask] = 0
    noisy.X = matrix
    return noisy


# Klein noisy
klein_noisy = inject_dropouts(klein_adata, dropout_rate, seed=5)
klein_noisy.layers["counts"] = klein_noisy.X.copy()
sc.pp.normalize_total(klein_noisy)
sc.pp.log1p(klein_noisy)
sc.tl.pca(klein_noisy, n_comps=50)
sc.tl.tsne(klein_noisy, n_pcs=50)
print(f"Klein noisy preprocessed: {klein_noisy.obsm['X_pca'].shape}")

# Zeisel noisy
zeisel_noisy = inject_dropouts(zeisel_adata, dropout_rate, seed=42)
zeisel_noisy.layers["counts"] = zeisel_noisy.X.copy()
sc.pp.normalize_total(zeisel_noisy)
sc.pp.log1p(zeisel_noisy)
sc.tl.pca(zeisel_noisy, n_comps=50)
sc.tl.tsne(zeisel_noisy, n_pcs=50)
print(f"Zeisel noisy preprocessed: {zeisel_noisy.obsm['X_pca'].shape}")


In [ ]:
# ============================================================
# Run all methods on noisy data
# ============================================================

# --- Klein noisy ---
klein_noisy_pca = klein_noisy.obsm["X_pca"][:, :50]

# KMeans
for k in klein_k_values:
    km = KMeans(n_clusters=k, random_state=5, n_init=10)
    klein_noisy.obs[f"kmeans_k{k}"] = km.fit_predict(klein_noisy_pca).astype(str)

# SC3
print("SC3 on Klein (noisy)...")
sc3s.tl.consensus(klein_noisy, n_clusters=klein_k_values)

# GAT
print("GAT on Klein (noisy)...")
klein_noisy_gat = train_gat(klein_noisy_pca, k=15, hidden_dim=64, embed_dim=32, epochs=200)
for k in klein_k_values:
    km = KMeans(n_clusters=k, random_state=5, n_init=10)
    klein_noisy.obs[f"gat_k{k}"] = km.fit_predict(klein_noisy_gat).astype(str)


# --- Zeisel noisy ---
zeisel_noisy_pca = zeisel_noisy.obsm["X_pca"][:, :50]

# KMeans
for k in zeisel_k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    zeisel_noisy.obs[f"kmeans_k{k}"] = km.fit_predict(zeisel_noisy_pca).astype(str)

# SC3
print("SC3 on Zeisel (noisy)...")
sc3s.tl.consensus(zeisel_noisy, n_clusters=zeisel_k_values)

# GAT
print("GAT on Zeisel (noisy)...")
zeisel_noisy_gat = train_gat(zeisel_noisy_pca, k=15, hidden_dim=64, embed_dim=32, epochs=200)
for k in zeisel_k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    zeisel_noisy.obs[f"gat_k{k}"] = km.fit_predict(zeisel_noisy_gat).astype(str)

print("\nAll noisy runs complete.")


## 7. Results

ARI (Adjusted Rand Index) comparison across all methods, both datasets, clean vs noisy.


In [ ]:
# ============================================================
# Klein results table
# ============================================================

true_klein = klein_adata.obs["cell_stage"]
klein_rows = []
for k in klein_k_values:
    sc3_col = f"sc3s_{k}"
    klein_rows.append({
        "k": k,
        "KMeans (Clean)": adjusted_rand_score(true_klein, klein_adata.obs[f"kmeans_k{k}"]),
        "SC3 (Clean)": adjusted_rand_score(true_klein, klein_adata.obs[sc3_col]),
        "GAT (Clean)": adjusted_rand_score(true_klein, klein_adata.obs[f"gat_k{k}"]),
        "KMeans (Noisy)": adjusted_rand_score(true_klein, klein_noisy.obs[f"kmeans_k{k}"]),
        "SC3 (Noisy)": adjusted_rand_score(true_klein, klein_noisy.obs[sc3_col]),
        "GAT (Noisy)": adjusted_rand_score(true_klein, klein_noisy.obs[f"gat_k{k}"]),
    })

klein_results = pd.DataFrame(klein_rows).set_index("k").round(4)
print("KLEIN (GSE65525) — ARI vs cell_stage")
print(klein_results)
print()

# ============================================================
# Zeisel results table
# ============================================================

true_zeisel = zeisel_adata.obs["cell_type"]
zeisel_rows = []
for k in zeisel_k_values:
    sc3_col = f"sc3s_{k}"
    zeisel_rows.append({
        "k": k,
        "KMeans (Clean)": adjusted_rand_score(true_zeisel, zeisel_adata.obs[f"kmeans_k{k}"]),
        "SC3 (Clean)": adjusted_rand_score(true_zeisel, zeisel_adata.obs[sc3_col]),
        "GAT (Clean)": adjusted_rand_score(true_zeisel, zeisel_adata.obs[f"gat_k{k}"]),
        "KMeans (Noisy)": adjusted_rand_score(true_zeisel, zeisel_noisy.obs[f"kmeans_k{k}"]),
        "SC3 (Noisy)": adjusted_rand_score(true_zeisel, zeisel_noisy.obs[sc3_col]),
        "GAT (Noisy)": adjusted_rand_score(true_zeisel, zeisel_noisy.obs[f"gat_k{k}"]),
    })

zeisel_results = pd.DataFrame(zeisel_rows).set_index("k").round(4)
print("ZEISEL (GSE60361) — ARI vs cell_type")
print(zeisel_results)


In [ ]:
# ============================================================
# Summary: best k per dataset, degradation under noise
# ============================================================

print("=" * 70)
print("FINAL SUMMARY — Best k per dataset")
print("=" * 70)

# Klein k=4 (4 differentiation stages)
print("\nKlein (k=4):")
print(f"  KMeans  | Clean: {adjusted_rand_score(true_klein, klein_adata.obs['kmeans_k4']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_klein, klein_noisy.obs['kmeans_k4']):.4f}")
print(f"  SC3     | Clean: {adjusted_rand_score(true_klein, klein_adata.obs['sc3s_4']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_klein, klein_noisy.obs['sc3s_4']):.4f}")
print(f"  GAT     | Clean: {adjusted_rand_score(true_klein, klein_adata.obs['gat_k4']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_klein, klein_noisy.obs['gat_k4']):.4f}")

# Zeisel k=7 (7 cell types)
print("\nZeisel (k=7):")
print(f"  KMeans  | Clean: {adjusted_rand_score(true_zeisel, zeisel_adata.obs['kmeans_k7']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_zeisel, zeisel_noisy.obs['kmeans_k7']):.4f}")
print(f"  SC3     | Clean: {adjusted_rand_score(true_zeisel, zeisel_adata.obs['sc3s_7']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_zeisel, zeisel_noisy.obs['sc3s_7']):.4f}")
print(f"  GAT     | Clean: {adjusted_rand_score(true_zeisel, zeisel_adata.obs['gat_k7']):.4f}"
      f"  Noisy: {adjusted_rand_score(true_zeisel, zeisel_noisy.obs['gat_k7']):.4f}")

# Degradation
print("\n" + "=" * 70)
print("DEGRADATION (Clean ARI - Noisy ARI) — lower is more robust")
print("=" * 70)
for name, true, clean_adata, noisy_adata, k in [
    ("Klein", true_klein, klein_adata, klein_noisy, 4),
    ("Zeisel", true_zeisel, zeisel_adata, zeisel_noisy, 7),
]:
    sc3_col = f"sc3s_{k}"
    km_deg = adjusted_rand_score(true, clean_adata.obs[f"kmeans_k{k}"]) - adjusted_rand_score(true, noisy_adata.obs[f"kmeans_k{k}"])
    sc3_deg = adjusted_rand_score(true, clean_adata.obs[sc3_col]) - adjusted_rand_score(true, noisy_adata.obs[sc3_col])
    gat_deg = adjusted_rand_score(true, clean_adata.obs[f"gat_k{k}"]) - adjusted_rand_score(true, noisy_adata.obs[f"gat_k{k}"])
    print(f"  {name:8s} | KMeans: {km_deg:+.4f}  SC3: {sc3_deg:+.4f}  GAT: {gat_deg:+.4f}")
